# 04 · DMP biomass yield vs the CPI water-balance yield

**Question.** How does a yield estimate built from **dry matter productivity** compare with the pipeline's CPI water-balance yield, against the same ground truth?

**How to run:** put the `planting_pipeline` folder on your Google Drive, run top-to-bottom, approve the Drive-mount and Earth-Engine prompts. Export cells start GEE tasks and return immediately; the scoring cells read the CSVs once the tasks finish (watch https://code.earthengine.google.com/tasks).

## Setup

### Stage 0 · Runtime

Installs the Earth Engine client, geemap, pandas, geopandas and scipy. `scipy` is the one that matters
here: every score in this notebook is a leave-one-out cross-validation with a paired bootstrap interval,
and both come from scipy.

**Expected output.** `installed.`

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas scipy 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine

`EE ready: ok`. The export cells below submit **batch tasks** and return immediately; the scoring cells
read the resulting CSVs. Between the two you have to wait, and you can close the browser while you do.
Watch the queue at code.earthengine.google.com/tasks.

**The export queue is per cloud project.** `ee-manzikye` has left batches in READY for hours. If the
tasks are not entering RUNNING within about 20 minutes, switch `PROJECT` to
`indigo-proxy-484220-q8` and resubmit rather than waiting.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Drive

`pipeline on path: ...`. The scoring scripts read and write `Cropyield-Data/` inside this folder, so the
notebook must `chdir` here for the relative paths to resolve.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Scoring convention used throughout
`n` is small (6–81 zones) in every test here, and a single 70/30 split at n≈45 has a **±0.10 t/ha standard deviation — larger than any effect measured**. So every test below uses **leave-one-out CV** (each free parameter refit on n−1) plus a **paired bootstrap** CI, and reports **Spearman** alongside MAE because rank skill is invariant to the yield ceiling Ym. Reporting a single split would have produced two false positives in this round.

## Data note — read before running
Copernicus DMP is **300 m or 1 km, not 250 m**, so it does not grid-align 1:1 with the 250 m products (everything here is reduced zonally, which sidesteps that). It is **not in the GEE catalog** — it needs a NetCDF download from land.copernicus.eu / Terrascope, ingest as an EE asset, then set `CGLS_DMP_ASSET` in `src/dmp_yield.py`.

Until then **MODIS MOD17A2H GPP stands in** (500 m / 8-day): GPP→NPP via CUE 0.45, C→dry matter ÷0.475. `--source modis|cgls` is a one-flag swap; everything downstream is identical. Treat MODIS magnitudes as provisional and rankings as more robust.

**Raw DM (kg/ha) is exported**, so harvest index, above-ground fraction and grain moisture are applied at scoring time and can be retuned with no re-export.

### Conversion
```
seasonal DM (kg/ha) = Σ (DMP_dekad × days_in_dekad)      # DMP is kg DM/ha/DAY
grain (kg/ha)       = DM × F_ABOVEGROUND × HARVEST_INDEX / (1 − moisture)
```
Maize defaults 0.80 / 0.45 / 0.135. Dekads are 8–11 days, not a flat 10, so a flat ×10 introduces a systematic few-percent bias — `dekad_to_t()` handles the real calendar, and time is indexed continuously so short-rains seasons maturing in the following year integrate without a day-of-year wrap.

### Stage 1 · Export the biomass route

**What runs.** Five zonal exports of seasonal dry matter, for Kenya long rains, Kenya short rains, the
2021 and 2022 crop-cut wards, and Ethiopia Meher.

**Raw dry matter in kg/ha is what is exported.** Harvest index, above-ground fraction and grain moisture
are applied at **scoring** time, so they can be retuned without re-exporting anything.

$$\text{seasonal DM}=\sum_t \mathrm{DMP}_t \times \text{days in dekad } t,\qquad
\text{grain}=\mathrm{DM}\,\frac{F_{\text{above}}\times HI}{1-\text{moisture}},$$

with maize defaults 0.80, 0.45 and 0.135. Dekads are 8 to 11 days, not a flat 10, and `dekad_to_t()`
uses the real calendar; a flat ×10 introduces a systematic error of a few percent, and time is indexed
continuously so a short-rains season maturing the following year integrates without a day-of-year wrap.

**The source is a stand-in and you must say so.** Copernicus DMP is 300 m or 1 km, is not in the Earth
Engine catalog, and needs a NetCDF download and ingest before `CGLS_DMP_ASSET` can be set. Until then
**MODIS MOD17A2H GPP stands in**, converted to net primary production with a carbon-use efficiency of
0.45 and to dry matter by dividing by 0.475. `--source modis|cgls` is a one-flag swap. Treat MODIS
**magnitudes as provisional and the rankings as more robust**, which is exactly what the results below
show.

In [ ]:
!python dmp_run.py --variant ke_long
!python dmp_run.py --variant ke_short
!python dmp_run.py --variant ke_ward_2021
!python dmp_run.py --variant ke_ward_2022
!python dmp_run.py --variant et_meher

### Stage 2 · Score the biomass route against the water-balance route

**Expected values.** Both routes against the same ground truth.

| Variant | n | DMP: MAE / bias / ρ | CPI: MAE / bias / ρ | Implied HI | DMP over-prediction |
|---|---|---|---|---|---|
| Kenya long 2024 | 43 | 1.09 / +1.04 / **+0.77** | 0.68 / +0.09 / +0.73 | 0.275 | 1.7× |
| Kenya short 2024 | 46 | 0.76 / +0.68 / **+0.64** | 0.52 / +0.06 / +0.59 | 0.285 | 1.6× |
| Kenya ward 2021 | 66 | 1.45 / +1.45 / **+0.49** | 1.25 / +1.22 / +0.15 | 0.066 | 7.3× |
| Kenya ward 2022, Kitui | 15 | 1.46 / +1.46 / **+0.41** | 0.54 / +0.54 / **−0.32** | 0.019 | 23.5× |
| Ethiopia Meher 2024 | 7 | **0.53 / −0.36 / +0.64** | 1.58 / +1.58 / **−0.40** | 0.468 | 0.8× |

**How to read it.** **DMP out-ranks the CPI water balance in all five variants.** Spearman is higher
every time, and the gap is widest exactly where CPI ranks **backwards**: Kitui 2022 at −0.32 and
Ethiopia Meher at −0.40. A negative rank correlation is worse than no information, and it is the single
most serious finding in this set.

**The level is a different matter.** DMP over-predicts by 1.6 to 23.5 times, and the implied harvest
index collapses to 0.019 in the Kitui wards, which is agronomically impossible. The cause is mixed
pixels: at 500 m the signal carries non-crop biomass. That is why notebook 05 uses a DMP **anomaly**
rather than raw biomass, since the contamination is largely static per pixel and differencing against
the pixel's own climatology cancels most of it.

**Ethiopia is the immediately actionable case.** DMP wins on every metric, and its implied harvest index
of 0.468 sits squarely in the agronomic range of 0.30 to 0.55. Ethiopia Meher is where the water balance
is least informative and the biomass route most defensible.

In [ ]:
!python dmp_score.py

## Result

| variant | n | DMP MAE / bias / ρ | CPI MAE / bias / ρ | implied HI | over-pred |
|---|---|---|---|---|---|
| KE Long 2024 | 43 | 1.09 / +1.04 / **+0.77** | 0.68 / +0.09 / +0.73 | 0.275 | 1.7× |
| KE Short 2024 | 46 | 0.76 / +0.68 / **+0.64** | 0.52 / +0.06 / +0.59 | 0.285 | 1.6× |
| KE ward 2021 | 66 | 1.45 / +1.45 / **+0.49** | 1.25 / +1.22 / +0.15 | 0.066 | 7.3× |
| KE ward 2022 Kitui | 15 | 1.46 / +1.46 / **+0.41** | 0.54 / +0.54 / **−0.32** | 0.019 | 23.5× |
| ET Meher 2024 | 7 | **0.53 / −0.36 / +0.64** | 1.58 / +1.58 / **−0.40** | 0.468 | 0.8× |

**DMP out-ranks the CPI water balance in all five** — Spearman higher every time, and the gap is widest exactly where CPI ranks *backwards* (Kitui 2022 −0.32, ET Meher −0.40).

**Ethiopia is immediately actionable:** DMP wins on every metric AND its implied harvest index (0.468) sits squarely in the agronomic 0.30–0.55 range — everything is internally consistent.

**Kenya's implied HI is the diagnostic, not a knob.** 0.275 / 0.285 are below the agronomic range; at ward level 0.066 and 0.019 are physically impossible. No harvest index reconciles those — the error is **mixed pixels**: smallholder 500 m pixels carry bush, weeds and intercrop biomass that grows whether or not the maize does. Hence DMP over-predicts the Kitui 2022 total failure by **23.5×**.

### How to register this — DMP measures the OUTCOME, not the CAUSE
DMP ranks yield better, but it **cannot attribute** *why* biomass is short (water? heat? pest? nutrient? late planting?), it is a **lagging** indicator (the shortfall appears *after* the stress, too late for anticipatory action), and it **cannot see failure** in smallholder mixed pixels. So it does **not** replace `S_water` / `S_heat`:

| role | use |
|---|---|
| causal attribution + lead time | WRSI / water balance (`S_water`), heat (`S_heat`) — keep |
| outcome ranking, late-season estimate | DMP |
| where DMP actually belongs in CPI | it is a better-calibrated **`S_veg`** — the existing greenness term — *not* a replacement for the stress terms |
| divergence as a diagnostic | DMP high + water balance stressed → suspect mask/irrigation; DMP low + balance fine → suspect heat, pest or nutrient |

The natural next test is therefore **swap or augment `S_veg` with DMP and re-score CPI** — not 'replace the water balance with DMP'.